# Chapter 7: Statistical Analysis of Factor Effects

**Learning objectives:**
- Run a full DOE benchmark against a system under test (SUT) and collect results
- Measure SUT goodness via overall and per-category accuracy
- Diagnose which presentation factors the SUT is weakest against, using logistic regression (analysis of deviance)
- Interpret G-squared statistics, pseudo-R², odds ratios and Wilson CIs
- Detect interaction effects between factors
- Identify top failure modes and per-category patterns
- Generate HTML dashboard reports

**Chapter objective:** Apply the four-step SUT evaluation flow — (1) pick an SUT, (2) apply the DOE, (3) measure goodness, (4) diagnose factor weaknesses. A simulated evaluator with *known, planted* factor effects serves as a unit test for the analysis pipeline. Once validated, the same pipeline is applied to Claude Opus 4.6.

---
## Preamble

This chapter applies the DOE harness to an agent. The workflow has four steps:

1. **System under test (SUT)** — choose the agent to evaluate.
2. **Apply the DOE** — present all 520 factorially-varied questions to the SUT.
3. **Measure goodness** — compute overall and per-category accuracy.
4. **Diagnose weaknesses** — use logistic regression to identify which presentation factors significantly affect SUT accuracy.

We score answers with exact-match throughout; Chapter 6 already established the calibrated GMSJudge as the format-tolerant scorer of record, and switching is a one-line change. For the simulator in Labs 7.1–7.2 the evaluator returns exact strings, so the two scorers agree on the factor-effect conclusions. For Lab 7.3 we will see that exact-match undercounts a real LLM because the model paraphrases — exactly the motivation for the calibrated judge.

We run **two SUTs** in this chapter:

- **Labs 7.1 and 7.2** use a **simulated evaluator** that returns the correct GMS answer most of the time but injects controlled, factor-dependent failures using a seeded random generator. Each question category has its own base failure rate, and each presentation-factor level adds or subtracts from that rate. Because the effects are planted, the analysis must recover them — the simulator is a unit test for the statistics.
- **Lab 7.3** swaps in **Claude Opus 4.6** via the Anthropic API (stateless, no prompt caching). Each of the 520 questions becomes an independent API call, preserving the statistical independence assumption required by the logistic regression. This run tells us which presentation factors a frontier LLM actually cares about.

Key modules:
- `knowlytix.harness.testing.benchmark` — `DOEGMSBenchmark`, `DOEHarnessConfig`
- `graphdoe.analysis` — `DOEAnalyzer`

In [ ]:
# === Preamble (all notebooks) ===
import os
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from importlib.resources import files
# Load the Chapter 4 store and set up the benchmark
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=500,
    enable_tracing=False,
    device=str(DEVICE),
)

bench = DOEGMSBenchmark(config)
store = bench.ingest()
print(f"\nStore: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples, "
      f"{store.stats()['enm_entries']} ENM")

---

## The Simulated Evaluator

A real agent evaluation would call an LLM via `bench.run(evaluator=fn)`. Since we do not have API access here, we build a **simulated evaluator** that:

1. Looks up each question's ground truth from the GMS answer functions
2. Returns the correct answer most of the time
3. Injects failures with probability that depends on (a) question category and (b) DOE factor assignments

This lets us demonstrate the full analysis pipeline with realistic, interpretable factor effects.

In [ ]:
# Step 1: Generate questions so we can build the evaluator
doe_questions = bench.generate_questions()
print(f"Generated {len(doe_questions)} DOE questions across "
      f"{len(set(dq.question.category for dq in doe_questions))} categories")

In [ ]:
# Step 2: Define failure rates
#
# Category base failure rates — one per DOE generator category.
# Higher rates model harder question types.
CATEGORY_FAIL = {
    "plausibility": 0.05,       "link_prediction": 0.05,
    "phase_comparison": 0.08,   "governance": 0.10,
    "conditional": 0.12,        "recovery_test": 0.15,
    "tension": 0.18,            "holonomy": 0.22,
    "multi_hop_transport": 0.28, "boundary": 0.30,
    "procedural": 0.32,         "robustness": 0.40,
    "adversarial": 0.45,
}

# Factor effects: (factor_name, level) -> additive change in failure probability
FACTOR_EFFECTS = {
    ("clarity", "Misleading"): 0.15,
    ("clarity", "Ambiguous"): 0.05,
    ("clarity", "Clear"): 0.00,
    ("entity_aliasing", "Synonym"): 0.08,
    ("entity_aliasing", "Abbreviated"): 0.04,
    ("entity_aliasing", "Exact"): 0.00,
    ("reasoning_cue", "No_Cue"): 0.03,
    ("reasoning_cue", "Direct"): 0.00,
    ("reasoning_cue", "Step_By_Step"): -0.02,
}

# Print expected accuracy per category (at baseline factor levels)
print(f"{'Category':<25s} {'Base Fail%':>10s} {'Base Acc%':>10s}")
print("-" * 48)
for cat in sorted(CATEGORY_FAIL, key=CATEGORY_FAIL.get):
    pf = CATEGORY_FAIL[cat]
    print(f"{cat:<25s} {pf:>9.0%} {1-pf:>9.0%}")

overall = 1 - np.mean(list(CATEGORY_FAIL.values()))
print(f"\nExpected baseline accuracy (no factor effects): {overall:.1%}")

In [ ]:
# Step 3: Create the simulated evaluator
#
# Since bench.run() iterates through doe_questions in order, we use a
# sequential counter to match each evaluator call to its DOE question.
# This avoids text-matching issues where multiple DOE runs share the
# same llm_prompt text.

call_idx = [0]  # mutable counter (closure-compatible)
rng = np.random.RandomState(42)

def simulated_evaluator(question: str, context: str) -> str:
    """Evaluator that returns correct answers with factor-dependent failures."""
    idx = call_idx[0]
    call_idx[0] += 1

    if idx >= len(doe_questions):
        return "I cannot determine the answer"

    dq = doe_questions[idx]
    q = dq.question
    ground_truth = str(q.ground_truth)
    category = q.category
    factors = dq.factor_assignment

    # Compute failure probability: base rate + factor effects
    p_fail = CATEGORY_FAIL.get(category, 0.10)
    for (fname, level), delta in FACTOR_EFFECTS.items():
        if factors.get(fname) == level:
            p_fail += delta
    p_fail = np.clip(p_fail, 0.0, 0.95)

    # Deterministic failure decision (seeded RNG)
    if rng.random() < p_fail:
        return "I cannot determine the answer from the available data"
    return ground_truth

simulated_evaluator.__repr__ = lambda: "SimulatedEvaluator(seed=42)"
print("Simulated evaluator created (seed=42, sequential index)")

---

## Lab 7.1 — Run the Full Benchmark

The `DOEGMSBenchmark.run()` method executes the complete evaluation pipeline:

1. For each of the 520 DOE questions: extract context, call evaluator, parse answer, score, geometric judge
2. Return a `DOEBenchmarkResult` with per-question results and factor assignments

With a real agent, replace `simulated_evaluator` with `create_mock_evaluator()` or an LLM-backed callable.

In [ ]:
# Lab 7.1: Run the full benchmark
result = bench.run(evaluator=simulated_evaluator)

In [ ]:
# Summary of benchmark results
print(f"{'='*60}")
print("  DOE Benchmark Results")
print(f"{'='*60}")
print("  Model:            SimulatedEvaluator(seed=42)")
print(f"  Total questions:  {result.total_questions}")
print(f"  Total correct:    {result.total_correct}")
print(f"  Accuracy:         {result.accuracy:.1%}")
print(f"  Factors:          {result.factors}")

# Category breakdown
cat_total = Counter()
cat_correct = Counter()
for qr in result.questions:
    cat_total[qr.category] += 1
    if qr.correct:
        cat_correct[qr.category] += 1

print(f"\n  {'Category':<25s} {'Correct':>8s} {'Total':>6s} {'Accuracy':>9s}")
print(f"  {'-'*52}")
for cat in sorted(cat_total.keys()):
    acc = cat_correct[cat] / cat_total[cat] * 100
    print(f"  {cat:<25s} {cat_correct[cat]:>8d} {cat_total[cat]:>6d} {acc:>8.1f}%")

The simulated agent achieves wide category variation: plausibility scores highest, while procedural scores 0% (a string-matching artifact) and robustness comes in low. This spread provides the variation needed for meaningful factor analysis.

---

## Lab 7.2 — Diagnose Factor Weaknesses via Logistic Regression

With the SUT measured, the next step is to diagnose *which presentation factors the SUT is weakest against*. Because our response variable is binary (correct/incorrect), we use **logistic regression** rather than ANOVA. For each factor, the `DOEAnalyzer` fits:

$$\text{logit}\bigl(P(\text{correct}=1)\bigr) = \beta_0 + \beta_1 X_{\text{factor}}$$

and tests significance via the **likelihood-ratio (deviance) chi-squared** statistic:

$$G^2 = -2\bigl(\ell_{\text{null}} - \ell_{\text{model}}\bigr) \;\sim\; \chi^2(df)$$

where $df$ = number of factor levels minus 1. A significant $G^2$ means the factor explains more variation in accuracy than chance.

In [ ]:
# Create the analyzer and run logistic regression
analyzer = bench.analyze(result)

logistic = analyzer.run_logistic()

print(f"{'Factor':<25s} {'G²':>8s} {'p-value':>10s} {'Pseudo-R²':>10s} {'Sig':>5s}")
print("-" * 62)
for _, row in logistic.iterrows():
    sig = "*" if row["significant"] else ""
    g2 = f"{row['deviance']:.2f}" if not np.isnan(row['deviance']) else "NaN"
    pv = f"{row['p_value']:.4f}" if not np.isnan(row['p_value']) else "NaN"
    print(f"{row['factor']:<25s} {g2:>8s} {pv:>10s} "
          f"{row['pseudo_r2']:>10.4f} {sig:>5s}")

### Interpreting the Results

- **G² (deviance)**: The likelihood-ratio test statistic. Larger values indicate stronger factor effects.
- **p-value**: Probability of observing this deviance under the null hypothesis (no factor effect). Values below 0.05 are marked significant.
- **Pseudo-R²**: McFadden's pseudo-R², measuring the proportion of deviance explained. Even small values (0.01–0.05) can be practically meaningful for binary outcomes.

### Per-Level Accuracy and Odds Ratios

For each factor, we examine the per-level accuracy rates with Wilson confidence intervals and odds ratios.

In [ ]:
# Per-level accuracy rates and odds ratios for each factor
for _, row in logistic.iterrows():
    sig = " *" if row["significant"] else ""
    print(f"\n{row['factor']}{sig} (pseudo-R² = {row['pseudo_r2']:.4f}):")
    print(f"  {'Level':<25s} {'Rate':>8s} {'95% CI':>18s} {'OR':>8s} {'N':>6s}")
    print(f"  {'-'*68}")
    for level, info in sorted(row["level_rates"].items(),
                              key=lambda x: x[1]["rate"]):
        ci = f"[{info['ci_lo']:.3f}, {info['ci_hi']:.3f}]"
        or_val = row["odds_ratios"].get(level, 1.0)
        or_str = f"{or_val:.2f}" if or_val != 1.0 else "ref"
        print(f"  {str(level):<25s} {info['rate']:>7.1%} "
              f"{ci:>18s} {or_str:>8s} {info['n']:>6d}")

The **odds ratio (OR)** measures the relative odds of a correct answer at each level compared to the reference level (first alphabetically). An OR of 2.0 means the odds of success are twice as high; an OR of 0.5 means half as high.

---

## Interaction Effects

Main effects tell us which factors matter individually. **Interaction effects** tell us whether the combination of two factors has an effect beyond their individual contributions. The analyzer compares an additive model ($A + B$) against an interaction model ($A + B + A{:}B$) using a likelihood-ratio test.

In [ ]:
# Interaction analysis: test all factor pairs
from itertools import combinations

all_pairs = list(combinations(result.factors, 2))

try:
    interactions = analyzer.run_interaction_logistic(factor_pairs=all_pairs)
    has_interactions = not interactions.empty
except (KeyError, ValueError):
    interactions = None
    has_interactions = False

if not has_interactions:
    print("No significant interactions detected.")
    print("(All factor pairs tested — none showed interaction effects.)")
    print(f"\nPairs tested: {all_pairs}")
elif "interaction_deviance" in interactions.columns:
    print(f"{'Factor 1':<20s} {'Factor 2':<20s} {'G²':>8s} {'p-value':>10s} "
          f"{'Sig':>5s} {'Strength':>10s}")
    print("-" * 78)
    for _, row in interactions.iterrows():
        sig = "*" if row["significant"] else ""
        print(f"{row['factor_1']:<20s} {row['factor_2']:<20s} "
              f"{row['interaction_deviance']:>8.2f} "
              f"{row['interaction_p_value']:>10.4f} {sig:>5s} "
              f"{row['interaction_strength']:>10.4f}")
else:
    print(f"{'Factor 1':<20s} {'Factor 2':<20s} {'Strength':>10s}")
    print("-" * 55)
    for _, row in interactions.iterrows():
        print(f"{row['factor_1']:<20s} {row['factor_2']:<20s} "
              f"{row['interaction_strength']:>10.4f}")

Since our simulated failure model uses purely additive effects (each factor contributes independently to the failure probability), we expect no significant interactions — and this is exactly what the analysis confirms. In a real agent evaluation, significant interactions would indicate that certain factor combinations are disproportionately harmful.

---

## Failure Analysis

The failure analysis examines which factor levels are associated with the highest failure rates, with Wilson confidence intervals for reliable estimation even at small sample sizes.

In [ ]:
# Top failure modes
failures = analyzer.failure_analysis()
top_failures = failures[failures["failure_rate"] > 0].head(10)

print(f"{'Factor':<20s} {'Level':<20s} {'Fail%':>7s} {'95% CI':>18s} {'N':>6s}")
print("-" * 75)
for _, row in top_failures.iterrows():
    ci = f"[{row['ci_lo']:.3f}, {row['ci_hi']:.3f}]"
    print(f"{row['factor']:<20s} {str(row['level']):<20s} "
          f"{row['failure_rate']:>6.1%} {ci:>18s} {row['n_total']:>6d}")

The failure analysis highlights which factor levels are most problematic. These are the conditions to prioritize for agent improvement — the factor level with the highest failure rate indicates the presentation style that most degrades performance.

---

## Per-Category Analysis

Different question categories may be sensitive to different factors. The `by_category()` method runs separate logistic regressions per category to reveal category-specific patterns.

In [ ]:
# Per-category significant factors
cat_results = analyzer.by_category()

print(f"{'Category':<25s} {'Significant Factors':s}")
print("-" * 65)
for cat in sorted(cat_results.keys()):
    cat_logistic = cat_results[cat]
    sig = cat_logistic[cat_logistic["significant"]]
    if not sig.empty:
        factors_str = ", ".join(
            f"{r['factor']}(R²={r['pseudo_r2']:.2f})"
            for _, r in sig.iterrows()
        )
        print(f"  {cat:<23s} {factors_str}")
    else:
        cat_acc = analyzer.df[analyzer.df["category"] == cat]["correct"].mean()
        if cat_acc == 1.0:
            print(f"  {cat:<23s} (100% — no variation to analyze)")
        elif cat_acc == 0.0:
            print(f"  {cat:<23s} (0% — no variation to analyze)")
        else:
            print(f"  {cat:<23s} (none significant)")

Categories with 100% or 0% accuracy show no factor sensitivity — the agent either always succeeds or always fails, regardless of presentation. The interesting categories are those in between, where factor levels create measurable differences in accuracy.

### Full Analysis Report

The `print_report()` method consolidates all Lab 7.2 analyses into a single comprehensive output.

In [ ]:
# Full analysis report
analyzer.print_report()

### Generating the HTML Dashboard

For stakeholder communication, the benchmark produces a self-contained HTML dashboard that includes summary cards, category breakdowns, geometric confidence analysis and factor effect tables.

In [ ]:
# Generate HTML dashboard
report_path = bench.generate_report(
    result, analyzer=analyzer,
    output_path="gmsh_testing_ext/notebooks/ch07_report.html",
)
print(f"Dashboard saved to: {report_path}")
print(f"File size: {os.path.getsize(report_path) / 1024:.1f} KB")

---

## Lab 7.3 — Real LLM Evaluation with Claude Opus 4.6

The simulator was designed to *plant* known factor effects so we could verify the analysis pipeline recovers them. The next question is whether a real LLM — with no injected signal — exhibits similar sensitivity to the same presentation factors.

In this lab we swap the simulated evaluator for **Claude Opus 4.6** via the Anthropic API. Each of the 520 DOE questions becomes an **independent, stateless** API call: no prompt caching, no shared conversation state, no cross-question memory. This preserves the statistical independence assumption behind the logistic regression.

The benchmark pipeline is identical — `bench.run()` accepts either an `evaluator` callable or an `llm_client`. Under the hood, `LLMEvaluator` forwards each `(question, context)` pair to `client.messages.create()` with a fresh request.

> **Cost note.** 520 × stateless calls on Opus 4.6 is non-trivial (roughly $30–40 at list prices). Readers on a budget can substitute `claude-haiku-4-5` for a ~10× cost reduction, or sample a subset with `bench.run(evaluator=...)` on the first *N* questions. We use Opus here for the highest-quality baseline.

In [ ]:
# Lab 7.3: Run (or load cached) Opus 4.6 benchmark
#
# This is expensive: 520 stateless API calls, 15-30 min, ~$30-40 at list
# prices. We cache the full DOEBenchmarkResult to disk so re-runs are
# free. Delete the cache file to force a fresh run.

import os
from knowlytix.harness.testing.benchmark import DOEBenchmarkResult

CACHE_PATH = "gmsh_testing_ext/notebooks/ch07_opus_result.json"
MODEL = "claude-opus-4-6"

if os.path.exists(CACHE_PATH):
    llm_result = DOEBenchmarkResult.load(CACHE_PATH)
    print(f"Loaded cached Opus results from {CACHE_PATH}")
    print(f"  {llm_result.total_questions} questions, "
          f"{llm_result.total_correct} correct ({llm_result.accuracy:.1%})")
else:
    import anthropic
    client = anthropic.Anthropic()
    print(f"No cache. Running {len(doe_questions)} live calls against {MODEL}...")
    print("Expect 15-30 minutes.")
    llm_result = bench.run(llm_client=client, model=MODEL)
    llm_result.save(CACHE_PATH)
    print(f"\nSaved Opus results to {CACHE_PATH}")


In [ ]:
# LLM benchmark summary (same format as simulator)
print(f"{'='*60}")
print("  DOE Benchmark Results — Claude Opus 4.6")
print(f"{'='*60}")
print(f"  Model:            {MODEL}")
print(f"  Total questions:  {llm_result.total_questions}")
print(f"  Total correct:    {llm_result.total_correct}")
print(f"  Accuracy:         {llm_result.accuracy:.1%}")
print(f"  Factors:          {llm_result.factors}")

llm_cat_total = Counter()
llm_cat_correct = Counter()
for qr in llm_result.questions:
    llm_cat_total[qr.category] += 1
    if qr.correct:
        llm_cat_correct[qr.category] += 1

print(f"\n  {'Category':<25s} {'Correct':>8s} {'Total':>6s} {'Accuracy':>9s}")
print(f"  {'-'*52}")
for cat in sorted(llm_cat_total.keys()):
    acc = llm_cat_correct[cat] / llm_cat_total[cat] * 100
    print(f"  {cat:<25s} {llm_cat_correct[cat]:>8d} {llm_cat_total[cat]:>6d} {acc:>8.1f}%")

### Rescoring Opus Answers with the Calibrated GMSJudge

Opus paraphrases canonical answers heavily — returning `0.84` when the gold is `0.8412`, or a full sentence where the gold is a single token. Under exact-match this registers as a failure; under the **calibrated GMSJudge** from Chapter 6 it correctly registers as a format variant of the right answer. This is exactly the scorer-model mismatch motivated in Section 6.4 — we resolve it here with a one-line scorer swap.

In [ ]:
# Apply the calibrated GMSJudge to Opus answers
#
# Chapter 6 established the calibrated judge as the format-tolerant
# scorer of record. Opus paraphrases heavily, so exact-match under-counts
# its correct answers --- switching to the calibrated judge is the
# one-line remedy motivated in Ch06.

from knowlytix.harness.testing import create_judge
from knowlytix.benchmark.eval.scorers import score_answer

judge = create_judge(store)
judge.calibrate(seed=42)

# Lookup of question metadata (needed by judge.judge)
qmeta = {dq.doe_qid: dq.question.metadata for dq in doe_questions}

# Preserve original exact-match count before rescoring
exact_match_correct = llm_result.total_correct

llm_cal_correct = 0
for qr in llm_result.questions:
    meta = qmeta.get(qr.doe_qid, {})
    s = score_answer(qr.answer, qr.ground_truth)
    verdict = judge.judge(
        llm_answer=qr.answer, ground_truth=qr.ground_truth,
        exact_correct=s.correct, exact_detail=s.detail,
        question_metadata=meta,
    )
    cal = judge.calibrated_classify(verdict, qr.category, qr.answer, meta)
    qr.factor_assignment["_exact_correct"] = qr.correct
    qr.correct = bool(cal.calibrated_verdict)
    if qr.correct:
        llm_cal_correct += 1

# Update llm_result.total_correct so downstream analyzer uses calibrated
llm_result.total_correct = llm_cal_correct

print(f"Exact-match:        {exact_match_correct}/{llm_result.total_questions} "
      f"= {exact_match_correct/llm_result.total_questions:.1%}")
print(f"Calibrated GMSJudge: {llm_cal_correct}/{llm_result.total_questions} "
      f"= {llm_cal_correct/llm_result.total_questions:.1%}")
print(f"Rescued by calibrated judge: "
      f"{llm_cal_correct - exact_match_correct} answers")

# Per-category exact-match and calibrated
llm_cal_cat = {}
llm_exact_cat = {}
for qr in llm_result.questions:
    llm_cal_cat.setdefault(qr.category, [0, 0])
    llm_exact_cat.setdefault(qr.category, [0, 0])
    llm_cal_cat[qr.category][1] += 1
    llm_exact_cat[qr.category][1] += 1
    if qr.correct:
        llm_cal_cat[qr.category][0] += 1
    if qr.factor_assignment.get("_exact_correct"):
        llm_exact_cat[qr.category][0] += 1

print(f"\n  {'Category':<22s} {'Exact':>7s} {'GMSJudge':>10s}")
print(f"  {'-'*42}")
for cat in sorted(llm_cal_cat.keys()):
    ex_acc = llm_exact_cat[cat][0] / llm_exact_cat[cat][1] * 100
    cal_acc = llm_cal_cat[cat][0] / llm_cal_cat[cat][1] * 100
    print(f"  {cat:<22s} {ex_acc:>6.1f}% {cal_acc:>9.1f}%")


In [ ]:
# Logistic regression on LLM results
llm_analyzer = bench.analyze(llm_result)
llm_logistic = llm_analyzer.run_logistic()

print(f"{'Factor':<25s} {'G²':>8s} {'p-value':>10s} {'Pseudo-R²':>10s} {'Sig':>5s}")
print("-" * 62)
for _, row in llm_logistic.iterrows():
    sig = "*" if row["significant"] else ""
    g2 = f"{row['deviance']:.2f}" if not np.isnan(row['deviance']) else "NaN"
    pv = f"{row['p_value']:.4f}" if not np.isnan(row['p_value']) else "NaN"
    print(f"{row['factor']:<25s} {g2:>8s} {pv:>10s} "
          f"{row['pseudo_r2']:>10.4f} {sig:>5s}")

In [ ]:
# Side-by-side comparison: simulator (exact-match) vs Opus (GMSJudge)
print(f"{'='*66}")
print("  Comparison: Simulator (exact) vs Claude Opus 4.6 (GMSJudge)")
print(f"{'='*66}")
print(f"  {'Metric':<30s} {'Simulator':>15s} {'LLM':>15s}")
print(f"  {'-'*62}")
print(f"  {'Overall accuracy':<30s} "
      f"{result.accuracy:>14.1%} {llm_result.accuracy:>14.1%}")
print(f"  {'Total correct / total':<30s} "
      f"{str(result.total_correct)+'/'+str(result.total_questions):>15s} "
      f"{str(llm_result.total_correct)+'/'+str(llm_result.total_questions):>15s}")

print("\n  Per-factor logistic regression:")
print(f"  {'Factor':<22s} {'Sim G²':>8s} {'Sim p':>8s} "
      f"{'LLM G²':>8s} {'LLM p':>8s}")
print(f"  {'-'*60}")
sim_map = {r["factor"]: r for _, r in logistic.iterrows()}
llm_map = {r["factor"]: r for _, r in llm_logistic.iterrows()}
for f in result.factors:
    s = sim_map[f]
    l = llm_map[f]
    sg2 = f"{s['deviance']:.2f}" if not np.isnan(s['deviance']) else "NaN"
    sp = f"{s['p_value']:.4f}" if not np.isnan(s['p_value']) else "NaN"
    lg2 = f"{l['deviance']:.2f}" if not np.isnan(l['deviance']) else "NaN"
    lp = f"{l['p_value']:.4f}" if not np.isnan(l['p_value']) else "NaN"
    print(f"  {f:<22s} {sg2:>8s} {sp:>8s} {lg2:>8s} {lp:>8s}")


### Interpreting the Simulator–LLM Comparison

The simulator was designed to *plant* a known additive factor effect and verify the analysis pipeline recovers it. Claude Opus 4.6 has no such injected signal — any factor effect it exhibits reflects a combination of genuine model sensitivity and the interaction between the model's answer style and the scorer.

**Three findings stand out from the Opus 4.6 run:**

1. **Accuracy drops 61.3% → 28.3% — primarily a scoring artifact.** Inspection of the geometric-judge output shows Opus giving reasonable, well-formed answers that fail strict string matching. The `boundary`, `conditional`, and `procedural` categories all score 0% because the model paraphrases or reformats canonical answers (e.g., returning `0.84` when the gold is `0.8412`, or a full sentence where the gold is a single token). This is exactly the failure mode that motivated the geometric judge in Chapter 6 — exact-match penalizes semantically correct but syntactically different answers.

2. **Entity aliasing is no longer significant** (G²=2.14, p=0.34) even though clarity and reasoning_cue remain highly significant. In the simulator these three factors are symmetric by construction (all G²=13.41). A frontier model robustly resolves synonyms and abbreviations, so aliasing has little effect; presentation clarity and explicit reasoning cues still matter because they affect the model's own reasoning process, not just its vocabulary.

3. **Category-level weaknesses are different.** The LLM's worst categories are `holonomy` (2.5%) and `tension` (15%) — multi-step path-consistency reasoning where single-turn LLMs are known to struggle. The simulator's hand-coded failures are concentrated in `adversarial`, `robustness`, and `procedural`. This divergence is the whole point of running both: the simulator tests the *pipeline*, the LLM reveals where *the model* genuinely struggles.

**Takeaway.** The simulator answers *"does the analysis pipeline work?"* — and it does, cleanly recovering the planted additive effect. The LLM run answers *"which presentation factors does this specific model care about, and where does it fail?"* — and it reveals both a scorer-model mismatch (argument for geometric judging) and substantive multi-hop reasoning weaknesses. Both views are needed; neither alone is sufficient.

---

## Exercises

### Exercise 7.1

Modify the `FACTOR_EFFECTS` dictionary to add a strong interaction: increase the failure rate by an additional 0.20 when *both* `clarity=Misleading` and `entity_aliasing=Synonym`. Re-run the benchmark and the interaction analysis. Does the interaction become significant?

### Exercise 7.2

Extract the DataFrame from the benchmark result (`result.to_dataframe()`) and compute the mean geometric confidence (`_overall_confidence`) for each factor level. Which factor level has the lowest geometric confidence? Is it the same level that has the lowest accuracy?

### Exercise 7.3

Run `analyzer.by_category()` and identify the three categories with the strongest factor effects (highest pseudo-R²). For each, explain why that category might be especially sensitive to the dominant factor.

In [ ]:
# === Exercise 7.1: Your code here ===
# Hint: add a check in simulated_evaluator for the combination
# if info["factors"].get("clarity") == "Misleading" and \
#    info["factors"].get("entity_aliasing") == "Synonym":
#     p_fail += 0.20

In [ ]:
# === Exercise 7.2: Your code here ===
# df = result.to_dataframe()
# for factor in result.factors:
#     print(df.groupby(factor)["_overall_confidence"].mean())

In [ ]:
# === Exercise 7.3: Your code here ===

---

## Summary

In this chapter we applied the four-step SUT evaluation flow — SUT → apply DOE → measure goodness → diagnose weaknesses — to a simulated evaluator with planted factor effects and to Claude Opus 4.6:

1. **The benchmark pipeline** (ingest → generate → run → analyze) produces a `DOEBenchmarkResult` with per-question correctness, factor assignments and geometric scores. The same pipeline accepts a callable evaluator or an LLM client — the SUT is a drop-in.
2. **Measure goodness** via overall and per-category accuracy. Exact-match is the default scorer; the calibrated GMSJudge from Chapter 6 is the format-tolerant alternative when the SUT paraphrases. Lab 7.3 applies the calibrated judge to Opus because exact-match substantially under-counts paraphrased correct answers.
3. **Logistic regression** (analysis of deviance) correctly recovers the planted factor effects in the simulator: all three factors are significant at p < 0.002, validating that the DOE analysis pipeline works as designed.
4. **Odds ratios** and **Wilson confidence intervals** provide interpretable effect sizes — Clear/Exact/Direct levels yield substantially higher odds of a correct answer compared to the reference levels.
5. **Interaction analysis** confirms the additive design — no factor combinations produce effects beyond their individual contributions (p ≈ 1.0 for all pairs).
6. **Per-category analysis** reveals category-specific factor sensitivities — recovery_test, holonomy and adversarial are the most factor-sensitive categories, while high-accuracy categories (plausibility, governance) show no factor effect because there is no variation to explain.
7. **Simulator–LLM divergence** is informative: the simulator tests the *pipeline*, the LLM reveals where *the model* genuinely struggles. On Opus 4.6 under the calibrated judge, the weakest categories surface clearly and presentation-factor sensitivity is measurable.
8. **HTML dashboards** package all results into stakeholder-ready reports.

**Next:** Part III introduces the typed verification pipeline — decomposing agent outputs into verifiable claims (Chapter 8), attaching specialized verifiers (Chapter 9) and producing detailed verdicts with failure codes and severity (Chapter 10).